# Visualizacao do arquivo `SIFCBR24.dbc`\n\nNotebook-base em Python para leitura, preparacao e visualizacao dos dados do arquivo `.dbc` incluido neste repositorio.\n\nA proposta inicial e investigar a distribuicao das notificacoes por Unidade Federativa e destacar onde ha maior concentracao de registros.

## 1. Instalacao das dependencias\n\nNo Google Colab, execute a celula abaixo uma vez para instalar a biblioteca que le arquivos `.dbc` do DATASUS.

In [ ]:
!pip -q install pysus seaborn

## 2. Importacao das bibliotecas

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nsns.set_theme(style="whitegrid", context="notebook")

## 3. Leitura do arquivo `.dbc`\n\nMantenha o arquivo `SIFCBR24.dbc` em `data/raw/`. No Colab, abra este notebook pelo GitHub ou faca upload do arquivo para a mesma estrutura de diretorios.

In [ ]:
CAMINHOS_DBC = [\n    Path("../data/raw/SIFCBR24.dbc"),\n    Path("data/raw/SIFCBR24.dbc"),\n    Path("SIFCBR24.dbc"),\n]\n\nARQUIVO_DBC = next((caminho for caminho in CAMINHOS_DBC if caminho.exists()), None)\n\nif ARQUIVO_DBC is None:\n    caminhos = ", ".join(str(caminho) for caminho in CAMINHOS_DBC)\n    raise FileNotFoundError(f"Arquivo nao encontrado. Caminhos testados: {caminhos}")\n\ntry:\n    from pysus.utilities.readdbc import read_dbc\nexcept ImportError as exc:\n    raise ImportError("Nao foi possivel importar read_dbc. Verifique se o pacote pysus foi instalado.") from exc\n\ndf = read_dbc(str(ARQUIVO_DBC), encoding="latin1")\n\nprint(df.shape)\ndf.head()

## 4. Inspecao inicial\n\nOs nomes dos campos indicam uma base de notificacoes relacionada a sifilis congenita. A etapa abaixo confere campos, tipos e alguns valores frequentes antes da visualizacao.

In [ ]:
display(df.info())\n\ncampos_interesse = [\n    "DT_NOTIFIC", "NU_ANO", "SG_UF_NOT", "SG_UF", "ID_MN_RESI",\n    "CS_SEXO", "CS_RACA", "ANT_IDADE", "ANT_PRE_NA", "ANTSIFIL_N",\n    "LAB_CONF", "TRA_ESQUEM", "ANT_TRATAD", "EVOLUCAO"\n]\n\nfor coluna in campos_interesse:\n    if coluna in df.columns:\n        print(f"\\n{coluna}")\n        print(df[coluna].value_counts(dropna=False).head(10))

## 5. Preparacao dos dados\n\nA visualizacao inicial agrega notificacoes por UF de residencia. Isso permite uma conclusao objetiva sobre concentracao territorial dos registros.

In [ ]:
dados = df.copy()\n\nuf_coluna = "SG_UF" if "SG_UF" in dados.columns else "SG_UF_NOT"\ndados[uf_coluna] = dados[uf_coluna].astype(str).str.strip()\ndados = dados[dados[uf_coluna].ne("")]\n\nnotificacoes_por_uf = (\n    dados\n    .groupby(uf_coluna, as_index=False)\n    .size()\n    .rename(columns={"size": "notificacoes", uf_coluna: "uf"})\n    .sort_values("notificacoes", ascending=False)\n)\n\nnotificacoes_por_uf["percentual"] = (\n    notificacoes_por_uf["notificacoes"] / notificacoes_por_uf["notificacoes"].sum() * 100\n)\n\ntop_ufs = notificacoes_por_uf.head(12).copy()\ntop_ufs

## 6. Visualizacao\n\nGrafico de barras horizontais com as UFs de maior numero de notificacoes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))\n\nplot_data = top_ufs.sort_values("notificacoes", ascending=True)\ncores = ["#4C78A8"] * len(plot_data)\ncores[-1] = "#F58518"\n\nax.barh(plot_data["uf"], plot_data["notificacoes"], color=cores)\n\nfor i, row in enumerate(plot_data.itertuples(index=False)):\n    ax.text(\n        row.notificacoes,\n        i,\n        f" {row.notificacoes:,} ({row.percentual:.1f}%)".replace(",", "."),\n        va="center",\n        fontsize=9\n    )\n\nax.set_title("UFs com mais notificacoes em SIFCBR24", fontsize=15, weight="bold")\nax.set_xlabel("Numero de notificacoes")\nax.set_ylabel("UF de residencia")\nax.spines[["top", "right", "left"]].set_visible(False)\nax.grid(axis="x", alpha=0.25)\nax.grid(axis="y", visible=False)\n\nplt.tight_layout()\n\nsaida_dir = Path("../outputs/figures")\nif not saida_dir.exists():\n    saida_dir = Path("outputs/figures")\nsaida_dir.mkdir(parents=True, exist_ok=True)\n\nplt.savefig(saida_dir / "visualizacao_sifcbr24.png", dpi=200, bbox_inches="tight")\nplt.show()

## 7. Proximos refinamentos\n\n- Validar a origem exata do arquivo e registrar o link original no relatorio.\n- Confirmar o significado dos codigos de UF, raca/cor, evolucao e tratamento no dicionario do SINAN/DATASUS.\n- Considerar uma segunda visualizacao cruzando UF com pre-natal, tratamento ou evolucao, caso a conclusao fique mais forte.\n- Usar a imagem `outputs/figures/visualizacao_sifcbr24.png` no `RELATORIO.md`.